# 🧪 生成式AI應用開發｜第04週
## Prompt Engineering 實務：把需求寫成可控的 Prompt

> **執行環境：Google Colab** ｜ **延續第 3 週的 OpenAI Responses API**

第 3 週你完成了第一個 API 呼叫並封裝成函式。本週把重點放在「怎麼把話講清楚」，
讓模型的輸出更穩定、更符合需求。你會學到：

1. 用 system prompt（instructions）設定角色與規則
2. 把複雜任務拆解成步驟
3. 用輸出限制控制格式、字數與語氣
4. 用 few-shot 範例引導輸出風格
5. 建立簡單的 prompt 測試與比較流程
6. 實作四大功能：摘要、分類、改寫、問答

官方文件入口：
- Prompt engineering: https://platform.openai.com/docs/guides/prompt-engineering
- Text generation: https://platform.openai.com/docs/guides/text

<div style="border-left:4px solid #1a73e8; padding:10px 12px; background:#eef4ff; margin:10px 0;">
  <font color="#1a73e8"><b>本週任務</b></font><br>
  把「摘要、分類、改寫、問答」四個功能，分別寫成可重複使用的函式，並學會用 prompt 控制輸出。
</div>


> **教師版**：本檔保留完整參考答案，可用於課堂示範、投影講解或課後核對。

## 🎯 本週學習目標

| # | 能力 | 對應後續課程 |
|---|------|-------------|
| 1 | 用 system prompt 設定角色與規則 | 所有 App 的回應品質 |
| 2 | 將複雜任務拆解成步驟 | Agent、多步驟工作流程 |
| 3 | 用輸出限制控制格式/字數/語氣 | Week 6 Structured Outputs |
| 4 | 用 few-shot 範例引導輸出 | 分類、抽取、風格控制 |
| 5 | 建立 prompt 測試與比較流程 | 專題調校與品質改善 |
| 6 | 實作摘要/分類/改寫/問答函式 | Week 7 Streamlit、Week 8 期中專題 |


---
## 🧰 第一節：環境準備（延續第 3 週）

本週直接沿用第 3 週的環境設定。Colab 每次重開 runtime 都需要重新安裝套件與設定 API key。


In [ ]:
# 安裝 OpenAI Python SDK
!pip install openai --quiet

# 匯入本週使用的模組
import os
import json
from openai import OpenAI

print("套件與模組準備完成")


### 🔐 設定 API Key

作法與第 3 週相同：在 Colab 左側 **Secrets** 面板新增 `OPENAI_API_KEY`，並打開存取權限。
本機執行則可用 `.env` + `python-dotenv` 或環境變數。絕對不要把金鑰寫死在程式碼中。


In [ ]:
# 從 Colab Secrets 讀取 API key，並轉成環境變數
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("已從 Colab Secrets 載入 API key")
except Exception as e:
    print("非 Colab 環境或尚未設定 Secrets：", e)
    print("請改用環境變數或 .env 設定 OPENAI_API_KEY")


In [ ]:
# 建立 client、選擇模型，並沿用第 3 週的安全呼叫函式
client = OpenAI()

# 課堂預設使用較低成本的 mini 模型；可用 OPENAI_MODEL 覆蓋。
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")


def ask_ai_safe(user_input, role="你是一位有幫助的助理", model=MODEL):
    """呼叫 Responses API，回傳 (是否成功, 文字)。role 即 system prompt。"""
    try:
        response = client.responses.create(
            model=model,
            instructions=role,
            input=user_input,
        )
        return True, response.output_text
    except Exception as e:
        return False, f"API 呼叫失敗：{e}"


print(f"準備完成，使用模型：{MODEL}")


---
## 🎭 第二節：System Prompt 與角色設定

`instructions`（system prompt）是給模型的「角色與規則」。
同一個問題，有沒有設定角色，回答風格差很多。下面直接比較。


In [ ]:
# 示範：同一個問題，比較「沒有角色設定」與「有角色設定」
question = "什麼是 API？"

print("===== 預設助理 =====")
ok, text = ask_ai_safe(question)
print(text)

print()
print("===== 面向小學生的老師 =====")
role = "你是一位老師，請用小學生也能懂的比喻回答，控制在 100 字內。"
ok, text = ask_ai_safe(question, role=role)
print(text)


### 🧠 2-1 好的 system prompt 有哪些要素？

<div style="border-left:4px solid #1a73e8; padding:10px 12px; background:#eef4ff; margin:10px 0;">
  <font color="#1a73e8"><b>寫 system prompt 的四個要點</b></font><br>
  1. <b>角色</b>：你是誰（例：資深客服、Python 助教）<br>
  2. <b>任務</b>：要做什麼（例：將文章摘要）<br>
  3. <b>規則/限制</b>：字數、格式、語氣、不能做的事<br>
  4. <b>輸出格式</b>：只輸出摘要、回 JSON、用條列…
</div>


---
## 🧭 第三節：任務拆解

遇到複雜任務時，把「一句話」拆成「幾個步驟」，模型比較不容易漏掉重點。
這也是後面 Agent（Week 14）的基礎能力。


In [ ]:
# 示範：把「幫我看這段程式」拆解成明確步驟
code_snippet = """
def average(nums):
    total = 0
    for n in nums:
        total += n
    return total / len(nums)

print(average([]))
"""

role = (
    "你是 Python 助教。請依序完成三個步驟，並用標題分段：\n"
    "1. 指出程式可能出錯的地方\n"
    "2. 說明為什麼會錯\n"
    "3. 給出修正後的程式碼"
)
ok, text = ask_ai_safe(code_snippet, role=role)
print(text)


---
## 📏 第四節：輸出限制與格式控制

想讓輸出好用，就要把「格式」講清楚：字數上限、只回清單、只回 JSON、不要開場白…
越明確的限制，後續程式越好接。


In [ ]:
# 示範：同一段需求，用不同輸出限制
topic = "請介紹生成式 AI 的優點。"

print("===== 限制：一句話（30 字內）=====")
ok, text = ask_ai_safe(topic, role="請用一句話回答，控制在 30 字內。")
print(text)

print()
print("===== 限制：3 個條列重點 =====")
ok, text = ask_ai_safe(topic, role="請用 3 個條列重點回答，每點不超過 15 字，不要開場白。")
print(text)

print()
print("===== 限制：只回 JSON =====")
ok, text = ask_ai_safe(topic, role='請只輸出 JSON，格式為 {"優點": ["...", "...", "..."]}，不要其他文字。')
print(text)


---
## 🧮 第五節：Few-shot 範例

有些任務用文字描述很難講清楚，不如直接「給幾個範例」。
這就是 few-shot：在 `input` 裡用 role/content 清單（就像第 3 週第六節）先給幾組「問→答」範例，再放真正要問的句子。


In [ ]:
# 示範：用 few-shot 做情緒分類（只回一個詞）
few_shot_input = [
    {"role": "developer", "content": "你是情緒分類器。只回答 正面、負面 或 中性 其中一個詞，不要其他說明。"},
    {"role": "user", "content": "這家店的服務很棒，我下次還會再來。"},
    {"role": "assistant", "content": "正面"},
    {"role": "user", "content": "等了一個小時菜還沒上，很失望。"},
    {"role": "assistant", "content": "負面"},
    {"role": "user", "content": "價格普通，份量也還可以。"},
]
response = client.responses.create(model=MODEL, input=few_shot_input)
print("模型判斷：", response.output_text)


### 🔁 5-1 什麼時候用 few-shot？

<div style="border-left:4px solid #1a73e8; padding:10px 12px; background:#eef4ff; margin:10px 0;">
  <font color="#1a73e8"><b>zero-shot vs few-shot</b></font><br>
  • <b>zero-shot</b>（不給範例）：任務簡單、規則容易用文字講清楚時。<br>
  • <b>few-shot</b>（給範例）：輸出格式/風格難描述、或模型常回錯時，範例比規則更有效。
</div>


---
## 🧪 第六節：Prompt 測試與比較

Prompt 不是寫一次就完美。實務上會先想好「測試輸入」，再比較不同 prompt 的輸出，選出最穩定的一個。


In [ ]:
# 建立一個小工具：用多組 role 跑同一個輸入，方便比較
def compare_prompts(user_input, roles):
    results = {}
    for name, role in roles.items():
        ok, text = ask_ai_safe(user_input, role=role)
        results[name] = text
    return results


sample = "請說明什麼是向量資料庫。"
roles = {
    "面向小學生": "請用小學生能懂的比喻解釋，100 字內。",
    "面向工程師": "你是資深工程師，請用精確的技術用語回答，並附一個實際例子。",
}
for name, out in compare_prompts(sample, roles).items():
    print(f"===== {name} =====")
    print(out)
    print()


---
## 📝 第七節：實作（1）摘要 summarize

把前面學的「角色 + 輸出限制」組合成一個可重複使用的摘要函式。


In [ ]:
def summarize(text, max_chars=100):
    role = (
        "你是專業的中文摘要助手。"
        "請將使用者提供的文章濃縮成重點摘要，"
        f"控制在 {max_chars} 字以內，只輸出摘要本身，不要加開場白。"
    )
    ok, result = ask_ai_safe(text, role=role)
    return result


article = (
    "生成式 AI 近年快速發展，從文字生成延伸到圖片、程式碼與語音。"
    "企業開始把大型語言模型導入客服、行銷與內部知識管理，"
    "但也面臨資料隱私、成本控制與輸出正確性等挑戰。"
)
print(summarize(article, max_chars=60))


---
## 🗂️ 第八節：實作（2）分類 classify

分類最重要的是「輸出限制」：只回類別名稱，方便程式後續處理。


In [ ]:
def classify(text, categories):
    cats = "、".join(categories)
    role = (
        "你是文字分類器。"
        f"請將使用者輸入分類成以下其中一類：{cats}。"
        "只輸出類別名稱，不要有其他文字。"
    )
    ok, result = ask_ai_safe(text, role=role)
    return result.strip()


CATEGORIES = ["帳務問題", "技術支援", "退貨退款", "其他"]
tickets = [
    "我的信用卡被重複扣款了，請幫我確認。",
    "App 打開一直閃退，該怎麼辦？",
    "這件衣服想退貨，可以嗎？",
]
for t in tickets:
    print(classify(t, CATEGORIES), "<-", t)


---
## ✍️ 第九節：實作（3）改寫 rewrite

改寫重點在「語氣控制」：保持原意，只變說法。


In [ ]:
def rewrite(text, style):
    role = (
        f"你是文字改寫助手。請將使用者的文字改寫成「{style}」的語氣，"
        "保持原意，只輸出改寫後的文字。"
    )
    ok, result = ask_ai_safe(text, role=role)
    return result


original = "喂，你的報告錯很多，重寫。"
for style in ["禮貌正式的職場語氣", "溫暖鼓勵的語氣"]:
    print(f"[{style}]")
    print(rewrite(original, style))
    print()


---
## ❓ 第十節：實作（4）問答 answer_question

這裡示範「依據參考資料回答」的問答，並用輸出限制防止模型編造（幻覺）。
這是第 11 週 RAG（文件問答）的雛形。


In [ ]:
def answer_question(context, question):
    role = (
        "你是問答助手。只能根據提供的參考資料回答問題，"
        "如果資料中沒有答案，請回答「資料中找不到答案」，不要自行編造。"
    )
    user_input = f"參考資料：\n{context}\n\n問題：{question}"
    ok, result = ask_ai_safe(user_input, role=role)
    return result


context = (
    "本課程每週三上午上課，共 18 週。"
    "期中為個人小專題，期末為小組專題發表。"
    "作業需透過 GitHub 繳交。"
)
print("Q1:", answer_question(context, "期末的評量方式是什麼？"))
print("Q2:", answer_question(context, "這門課的老師是誰？"))


---
## 🧪 第十一節：課堂練習

把本週的 system prompt、任務拆解、輸出限制、few-shot 串起來。

### 📝 練習 A：會議記錄整理助手

1. 設計 `role`，讓 AI 扮演「會議記錄整理助手」
2. 任務拆解：要求依序輸出三個區塊——一句話摘要、分工清單、重要時程
3. 只根據提供的內容整理，不要補充沒提到的資訊

<div style="border-left:4px solid #188038; padding:10px 12px; background:#e6f4ea; margin:10px 0;">
  <font color="#188038"><b>課堂練習</b></font><br>
  先完成練習 A，再依時間進行練習 B 與選做練習 C。
</div>


In [ ]:
# 練習 A：組合 system prompt + 任務拆解
meeting_note = (
    "今天討論期末專題分組。小明負責前端，小華負責 API 串接，"
    "小美負責簡報。下週三前要交出雛形，下下週開始測試。"
)
role = (
    "你是會議記錄整理助手。請依序輸出三個區塊："
    "1) 一句話摘要 2) 分工清單（條列） 3) 重要時程（條列）。"
    "只根據使用者提供的內容整理，不要補充沒提到的資訊。"
)
ok, result = ask_ai_safe(meeting_note, role=role)
print(result)


---
### 🔁 練習 B：用 few-shot 提升分類穩定度

仿照第五節的 few-shot 示範，製作一個「垃圾郵件分類器」（只回 垃圾 或 正常），
至少提供 2 組範例對（user / assistant）。


In [ ]:
# 練習 B：few-shot 垃圾郵件分類
few_shot = [
    {"role": "developer", "content": "你是垃圾郵件分類器。只回答 垃圾 或 正常。"},
    {"role": "user", "content": "恭喜中獎！點此連結領取百萬獎金。"},
    {"role": "assistant", "content": "垃圾"},
    {"role": "user", "content": "下週一的會議改到下午三點，請注意。"},
    {"role": "assistant", "content": "正常"},
    {"role": "user", "content": "限時優惠，輸入卡號立即免費領取。"},
]
response = client.responses.create(model=MODEL, input=few_shot)
print("模型判斷：", response.output_text)


---
### 🎨 練習 C（選做）：Prompt 測試

用不同的 system prompt 摘要同一段文字，用 `compare_prompts` 觀察哪一種最符合需求。


In [ ]:
# 練習 C（選做）：用 compare_prompts 比較不同摘要 prompt
text = (
    "Streamlit 是一個 Python 函式庫，能用少量程式碼把資料腳本變成互動式網頁應用。"
    "它適合快速打造 Demo，但在高流量正式服務上仍有限制。"
)
prompt_variants = {
    "一句話": "請用一句話摘要，控制在 30 字內。",
    "條列 2 點": "請用 2 個條列重點摘要，每點不超過 20 字。",
    "給主管看": "你是助理，請摘要成主管能快速決策的重點，語氣專業。",
}
for name, out in compare_prompts(text, prompt_variants).items():
    print(f"===== {name} =====")
    print(out)
    print()


---
## ✅ 本週小結

| 技能 | 本週學了什麼 | 下週用在哪裡 |
|------|-------------|-------------|
| System prompt / 角色 | 用 instructions 設定角色、任務、規則 | 每一個 App 的回應品質 |
| 任務拆解 | 把複雜任務拆成步驟 | Week 14 Agent |
| 輸出限制 | 控制字數、格式、只回 JSON | Week 6 Structured Outputs |
| Few-shot | 用範例引導輸出 | 分類、抽取、風格控制 |
| Prompt 測試 | 比較多組 prompt 選最穩定 | 專題調校 |
| 四大功能 | summarize / classify / rewrite / answer_question | Week 7 Streamlit、Week 8 期中專題 |

<div style="border-left:4px solid #1a73e8; padding:10px 12px; background:#eef4ff; margin:10px 0;">
  <font color="#1a73e8"><b>下週預告</b></font><br>
  Week 5 將進入多輪對話、Streaming 與簡易聊天機器人，把本週的函式接上對話狀態。
</div>
